# MMELON Model Fine-Tuning

Fine-tune the MMELON (Multi-Modal Encoder for Learning on Molecules) model on hit prediction data.

## Overview

This notebook demonstrates how to fine-tune MMELON for binary classification tasks in drug discovery:

**What is MMELON?**
- Multi-modal molecular encoder combining multiple molecular representations
- Integrates graph neural networks, fingerprints, and other molecular features
- Pre-trained on large-scale molecular datasets

**Workflow:**
1. **Setup & Configuration** - Environment setup and hyperparameter configuration
2. **Data Preparation** - Load screening data and create data loaders
3. **Model Initialization** - Load pre-trained MMELON and configure prediction head
4. **Training Setup** - Configure PyTorch Lightning trainer
5. **Fine-Tuning** - Train the model on your dataset
6. **Evaluation** - Assess model performance and save results

**Use Cases:**
- Hit prediction from screening campaigns
- Activity classification (active/inactive)
- Lead optimization prioritization

In [1]:
IN_COLAB= False
import os
run_on_wdr91 = False
USER = os.getenv("USER")  
  
if IN_COLAB:
    MODELS_DIR = "/content/models/mmelon"
    DEMO_SIZE = 100
else:
    USER = os.getenv("USER")
    MODELS_DIR = f"/home/jupyter/repos/mmelon/biomed-multi-view/model"
    #DEMO_SIZE = 100 # None  !!!!!!!!!!!CHANGE
    DEMO_SIZE = None

if run_on_wdr91:
    NAME = "wdr91_asms"
    if IN_COLAB:
        DATA_PATH = "/content/data"
    else:
        # DATA_PATH = "/proj/ligand_ai/datasets_manager/processed/splits/wdr91_ASMS_train_val_v1"
        
        DATA_PATH = f"/home/jupyter/repos/mmelon/biomed-multi-view/notebooksdata/finetuning/wdr91_ASMS_train_val_v1"
        # DATA_PATH = f"/notebooks data/finetuning/wdr91_ASMS_train_val_v1"
else:
    NAME = "pgk2_del_cdd"
    if IN_COLAB:
        DATA_PATH = "/content/data"
    else:
        # DATA_PATH = f"/proj/ligand_ai/users/{USER}/data/PGK2_DEL_cdd_to_creative"
        DATA_PATH = f"/home/jupyter/repos/mmelon/biomed-multi-view/notebooksdata/finetuning/PGK2_DEL_cdd_to_creative"

MODEL_DIR = os.path.join(MODELS_DIR, NAME)
if DEMO_SIZE is not None:
    MODEL_DIR += f"_demo_{DEMO_SIZE}"
    DEMO_DATA_PATH = DATA_PATH + f"_demo_{DEMO_SIZE}" 
    os.makedirs(DEMO_DATA_PATH, exist_ok=True)    

# ============================================================================
# MODEL CONFIGURATION
# ============================================================================
BASE_MODEL = "ibm-research/biomed.sm.mv-te-84m"  # Hugging Face model name

# ============================================================================
# TRAINING HYPERPARAMETERS
# ============================================================================
#EPOCHS = 1 # 10  # Number of training epochs Change here !!!!!!!!!!!!!!!!!!
EPOCHS = 50
BATCH_SIZE = 16  # Training batch size
LEARNING_RATE = 2e-5  # Learning rate
WEIGHT_DECAY = 0.01  # Weight decay for regularization
SEED = 42  # Random seed for reproducibility

# ============================================================================
# DEVICE CONFIGURATION
# ============================================================================
DEVICE = None  # None for auto-detection, or specify 'cuda' or 'cpu'

print("Configuration loaded successfully!")
print(f"Output model directory: {MODEL_DIR}")
print(f"Base Model: {BASE_MODEL}")
print(f"Training epochs: {EPOCHS}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Learning rate: {LEARNING_RATE}")



Configuration loaded successfully!
Output model directory: /home/jupyter/repos/mmelon/biomed-multi-view/model/pgk2_del_cdd
Base Model: ibm-research/biomed.sm.mv-te-84m
Training epochs: 50
Batch size: 16
Learning rate: 2e-05


## 1. Setup and configuration

Edit these paths and hyperparameters according to your needs.

## 2. Load and Prepare Data

Load the dataset splits and prepare them for training.

In [2]:
import numpy as np
import pandas as pd
def reduce_data_for_demo(df: pd.DataFrame, demo_size: int, label_column: str, seed: int=42)-> pd.DataFrame:
          
    # Sample at most demo_size samples per class (label=0 and label=1)
    df_list = []
    for label in df[label_column].unique():
        df_label = df[df[label_column] == label]
        n = len(df_label)
        if n > demo_size:
            df_label = df_label.sample(n=demo_size, random_state=seed)
            print(f"Class {label}: {n} samples reduced to {len(df_label)} samples for demo.")
        else:
            print(f"Class {label}: {n} samples (no reduction needed for demo).")
        df_list.append(df_label)

    # Combine and shuffle
    df = pd.concat(df_list).sample(frac=1, random_state=seed).reset_index(drop=True)
    
    return df

In [3]:
import pandas as pd
import numpy as np
# Load split data

split_path = DATA_PATH
print (split_path)

print(f"Loading split from: {split_path}")

data_dir = DATA_PATH if DEMO_SIZE is None else DEMO_DATA_PATH
for split in ['train', 'val', 'test']:
    df = pd.read_csv(f"{split_path}/{split}.csv")
    
    if DEMO_SIZE is not None:
        original_size = len(df)
        df = reduce_data_for_demo(df, demo_size=DEMO_SIZE, label_column='label', seed=42)
        print(f"***** DEMO MODE: Reduced {split} set from {original_size} to {len(df)} samples (max {DEMO_SIZE} per class)")
    
    output_file = f"{data_dir}/data_{split}.csv"
    df.to_csv(output_file, index=False)
    print(f"Saved {output_file} for MMELON fine-tuning.")
    print(f"***  {split}: {len(df):,} samples ({df['label'].mean()*100:.2f}% positive)")

/home/jupyter/repos/mmelon/biomed-multi-view/notebooksdata/finetuning/PGK2_DEL_cdd_to_creative
Loading split from: /home/jupyter/repos/mmelon/biomed-multi-view/notebooksdata/finetuning/PGK2_DEL_cdd_to_creative
Saved /home/jupyter/repos/mmelon/biomed-multi-view/notebooksdata/finetuning/PGK2_DEL_cdd_to_creative/data_train.csv for MMELON fine-tuning.
***  train: 24,611 samples (7.87% positive)
Saved /home/jupyter/repos/mmelon/biomed-multi-view/notebooksdata/finetuning/PGK2_DEL_cdd_to_creative/data_val.csv for MMELON fine-tuning.
***  val: 2,735 samples (7.86% positive)
Saved /home/jupyter/repos/mmelon/biomed-multi-view/notebooksdata/finetuning/PGK2_DEL_cdd_to_creative/data_test.csv for MMELON fine-tuning.
***  test: 14,748 samples (9.95% positive)


## 3. Prepare Data Loaders

In [4]:
import pytorch_lightning as pl
from torch.utils.data import DataLoader

from bmfm_sm.core.data_modules.namespace import LateFusionStrategy, TaskType
from bmfm_sm.predictive.modules.finetune_lightning_module import FineTuneLightningModule
from bmfm_sm.predictive.data_modules.multimodal_finetune_dataset import MultiModalFinetuneDataPipeline
from bmfm_sm.api.smmv_api import SmallMoleculeMultiViewModel

2026-03-18 00:54:59,987 - rdkit - INFO - instance-20260302-162623:140392856098624:0:0 - Enabling RDKit 2022.09.5 jupyter extensions


In [5]:
# Set random seed for reproducibility
pl.seed_everything(SEED)

# Create output directory
os.makedirs(MODEL_DIR, exist_ok=True)

print("="*80)
print("INITIALIZING MMELON MODEL")
print("="*80)

# Dataset arguments for MMELON framework
dataset_args = {
    'task_type': TaskType.CLASSIFICATION,
    'num_tasks': 1,  # Single task for binary classification
    'modalities': ['TEXT_MODEL', 'IMAGE_MODEL', 'GRAPH_2D_MODEL'],
    'smiles_col': 'smiles',
    'label_cols': ['label'],
    'split_col': None
}

# Create data modules
print(f"Creating data modules using data in {data_dir}...")
train_dataset = MultiModalFinetuneDataPipeline(
    data_dir=str(data_dir),
    dataset_args=dataset_args,
    stage='train'
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=train_dataset.collate_fn,
    num_workers=4
)

val_dataset = MultiModalFinetuneDataPipeline(
    data_dir=str(data_dir),
    dataset_args=dataset_args,
    stage='val'
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=val_dataset.collate_fn,
    num_workers=4
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")

Seed set to 42
2026-03-18 00:55:00,518 - root - INFO - instance-20260302-162623:140392856098624:0:0 - Working on task TaskType.CLASSIFICATION
2026-03-18 00:55:00,519 - root - INFO - instance-20260302-162623:140392856098624:0:0 - modalities running in data pipeline TEXT_MODEL,IMAGE_MODEL,GRAPH_2D_MODEL


INITIALIZING MMELON MODEL
Creating data modules using data in /home/jupyter/repos/mmelon/biomed-multi-view/notebooksdata/finetuning/PGK2_DEL_cdd_to_creative...


2026-03-18 00:55:01,292 - root - INFO - instance-20260302-162623:140392856098624:0:0 - Working on task TaskType.CLASSIFICATION
2026-03-18 00:55:01,292 - root - INFO - instance-20260302-162623:140392856098624:0:0 - modalities running in data pipeline TEXT_MODEL,IMAGE_MODEL,GRAPH_2D_MODEL


Train batches: 1539
Val batches: 171


## 4. Initialize MMELON Model

In [6]:
FUSION_STRATEGY = LateFusionStrategy.ATTENTIONAL

# Model parameters
model_params = {
    'agg_arch': FUSION_STRATEGY.value[0],
    'agg_gate_input': FUSION_STRATEGY.value[1],
    'agg_weight_freeze': FUSION_STRATEGY.value[2],
    'inference_mode': False
}

# Finetuning arguments
finetuning_args = {
    'weight_freeze': 'unfrozen',
    'initialization': 'default',
    'head_arch': 'mlp',
    'use_norm': True,
    'head_dropout': 0.2
}

# Create Lightning module
print("Creating Lightning module...")
lightning_module = FineTuneLightningModule(
    base_model_class='bmfm_sm.predictive.modules.smmv_model.SmallMoleculeMultiView',
    model_params=model_params,
    task_type='classification',
    num_tasks=1,
    checkpoint_path=None,
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    finetuning_args=finetuning_args
)

# Load pretrained weights from HuggingFace
print(f"Loading pretrained model from HuggingFace: {BASE_MODEL}")
pretrained_model = SmallMoleculeMultiViewModel.from_pretrained(
    FUSION_STRATEGY,
    model_path=BASE_MODEL,
    huggingface=True
)

lightning_module.model.load_state_dict(pretrained_model.state_dict(), strict=False)

print("Model initialized successfully!")

2026-03-18 00:55:01,800 - root - INFO - instance-20260302-162623:140392856098624:0:0 - Initializing model with params {'agg_arch': 'coeff_mlp', 'agg_gate_input': 'coeff_mlp', 'agg_weight_freeze': None, 'inference_mode': False}


Creating Lightning module...


/opt/conda/envs/biomed-multiview/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/conda/envs/biomed-multiview/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
2026-03-18 00:55:03,264 - root - INFO - instance-20260302-162623:140392856098624:0:0 - Using coeff_mlp architecture for aggregator
2026-03-18 00:55:03,266 - root - INFO - instance-20260302-162623:140392856098624:0:0 - dim_list [512, 512, 768] for aggregator
2026-03-18 00:55:03,297 - root - INFO - instance-20260302-162623:140392856098624:0:0 - Initialized the model with random weights as checkpoint_path was specified as None
2026-03-18 00:55:03

Loading pretrained model from HuggingFace: ibm-research/biomed.sm.mv-te-84m


2026-03-18 00:55:04,909 - root - INFO - instance-20260302-162623:140392856098624:0:0 - Using coeff_mlp architecture for aggregator
2026-03-18 00:55:04,910 - root - INFO - instance-20260302-162623:140392856098624:0:0 - dim_list [512, 512, 768] for aggregator
2026-03-18 00:55:10,345 - root - INFO - instance-20260302-162623:140392856098624:0:0 - in train False setting deterministic_eval = True


Model initialized successfully!


## 5. Configure Trainer

In [7]:
from pytorch_lightning.callbacks import ModelCheckpoint

# Setup callbacks
callbacks = [
    ModelCheckpoint(
        dirpath=MODEL_DIR,
        filename='mmelon-{epoch:02d}-{val_loss:.2f}',
        save_top_k=3,
        monitor='val_loss',
        mode='min',
        save_last=True,
        every_n_epochs=1
    )
]

# Create trainer
trainer = pl.Trainer(
    max_epochs=EPOCHS,
    callbacks=callbacks,
    default_root_dir=MODEL_DIR,
    accelerator='auto',
    devices=1,
    log_every_n_steps=10,
    val_check_interval=1.0,
    enable_checkpointing=True,
    enable_model_summary=True
)


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
`Trainer(val_check_interval=1.0)` was configured so validation will run at the end of the training epoch..


## 6. Fine-Tune Model

In [ ]:
%%time

print("="*80)
print("STARTING TRAINING")
print("="*80)

trainer.fit(lightning_module, train_loader, val_loader)

print("="*80)
print("TRAINING COMPLETE!")
print("="*80)

You are using a CUDA device ('NVIDIA L4') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision


STARTING TRAINING


/opt/conda/envs/biomed-multiview/lib/python3.11/site-packages/pytorch_lightning/callbacks/model_checkpoint.py:881: Checkpoint directory /home/jupyter/repos/mmelon/biomed-multi-view/model/pgk2_del_cdd exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name      | Type                    | Params | Mode  | FLOPs
----------------------------------------------------------------------
0 | model     | SmallMoleculeMultiView  | 84.1 M | train | 0    
1 | pred_head | MultiTaskPredictionHead | 461 K  | train | 0    
2 | criterion | BCEWithLogitsLoss       | 0      | train | 0    
----------------------------------------------------------------------
84.6 M    Trainable params
0         Non-trainable params
84.6 M    Total params
338.207   Total estimated model params size (MB)
510       Modules in train mode
0         Modules in eval mode
0         Total Flops


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

2026-03-18 00:55:13,429 - root - INFO - instance-20260302-162623:140392856098624:0:0 - in train False setting deterministic_eval = True
/opt/conda/envs/biomed-multiview/lib/python3.11/site-packages/pytorch_lightning/utilities/data.py:79: Trying to infer the `batch_size` from an ambiguous collection. The batch size we found is 16. To avoid any miscalculations, use `self.log(..., batch_size=batch_size)`.


Training: |          | 0/? [00:00<?, ?it/s]

/opt/conda/envs/biomed-multiview/lib/python3.11/site-packages/fast_transformers/feature_maps/fourier_features.py:37: UserWarning: torch.qr is deprecated in favor of torch.linalg.qr and will be removed in a future PyTorch release.
The boolean parameter 'some' has been replaced with a string parameter 'mode'.
Q, R = torch.qr(A, some)
should be replaced with
Q, R = torch.linalg.qr(A, 'reduced' if some else 'complete') (Triggered internally at ../aten/src/ATen/native/BatchLinearAlgebra.cpp:2426.)
  Q, _ = torch.qr(block)


In [ ]:
os.listdir(MODEL_DIR)